# 02. YOLOv10 Model Training

This notebook handles the full training pipeline for the **Fruit & Vegetable Disease Detection** model using YOLOv10.

**Workflow:**
1. Environment & GPU check
2. Dataset verification
3. Model training with tuned hyperparameters
4. Results visualization (loss curves, mAP, confusion matrix)
5. Export best weights for deployment

## 1. Imports & Path Configuration

In [1]:
import os
import torch
import shutil
from pathlib import Path
from ultralytics import YOLO
from IPython.display import Image, display

# ==========================================
# PATH CONFIGURATION
# ==========================================
PROJECT_ROOT = Path("..").resolve()
DATASET_YAML = PROJECT_ROOT / "data" / "processed" / "dataset.yaml"
PRETRAINED_DIR = PROJECT_ROOT / "models" / "pretrained"
WEIGHTS_DIR = PROJECT_ROOT / "models" / "weights"

# Create directories if they don't exist
PRETRAINED_DIR.mkdir(parents=True, exist_ok=True)
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root : {PROJECT_ROOT}")
print(f"Dataset YAML : {DATASET_YAML}")
print(f"Pretrained dir: {PRETRAINED_DIR}")
print(f"Weights dir   : {WEIGHTS_DIR}")

Project root : D:\in-class\Advance_AI2\advanced_ai_project
Dataset YAML : D:\in-class\Advance_AI2\advanced_ai_project\data\processed\dataset.yaml
Pretrained dir: D:\in-class\Advance_AI2\advanced_ai_project\models\pretrained
Weights dir   : D:\in-class\Advance_AI2\advanced_ai_project\models\weights


## 2. Environment & GPU Check

In [2]:
print("=" * 50)
print("ENVIRONMENT INFO")
print("=" * 50)
print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    DEVICE = "cuda"
    print(f"GPU device      : {torch.cuda.get_device_name(0)}")
    print(f"GPU memory      : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    DEVICE = "cpu"
    print("⚠️  No GPU detected. Training will run on CPU (much slower).")

print(f"Selected device : {DEVICE}")
print("=" * 50)

ENVIRONMENT INFO
PyTorch version : 2.6.0+cu124
CUDA available  : True
GPU device      : NVIDIA GeForce RTX 4070
GPU memory      : 12.0 GB
Selected device : cuda


## 3. Dataset Verification

Verify that the processed dataset from `01_data_annotation.ipynb` exists and has the expected structure.

In [3]:
def verify_dataset(yaml_path: Path) -> bool:
    """Verify that the dataset exists and report split statistics."""
    if not yaml_path.exists():
        print(f"❌ Dataset config not found: {yaml_path}")
        print("   Please run 01_data_annotation.ipynb first.")
        return False

    processed_dir = yaml_path.parent
    print("=" * 50)
    print("DATASET VERIFICATION")
    print("=" * 50)

    total = 0
    for split in ["train", "val", "test"]:
        img_dir = processed_dir / "images" / split
        lbl_dir = processed_dir / "labels" / split

        n_images = len(list(img_dir.glob("*"))) if img_dir.exists() else 0
        n_labels = len(list(lbl_dir.glob("*.txt"))) if lbl_dir.exists() else 0
        total += n_images

        status = "✅" if n_images > 0 and n_images == n_labels else "⚠️"
        print(f"  {status} {split:5s} -> {n_images:5d} images | {n_labels:5d} labels")

    print(f"\n  Total images: {total}")
    print("=" * 50)
    return total > 0

dataset_ok = verify_dataset(DATASET_YAML)
assert dataset_ok, "Dataset is missing or empty. Run 01_data_annotation.ipynb first."

DATASET VERIFICATION
  ⚠️ train -> 23414 images | 22727 labels
  ⚠️ val   ->  4380 images |  4356 labels
  ⚠️ test  ->  1483 images |  1478 labels

  Total images: 29277


## 4. Training Configuration

Define all hyperparameters in one place for easy tuning.

In [ ]:
# ==========================================
# TRAINING HYPERPARAMETERS
# ==========================================
YOLO_MODEL = str(PRETRAINED_DIR / "yolov10n.pt")  # Base model: nano variant for faster training
EPOCHS = 100                 # Number of training epochs
IMG_SIZE = 800              # Input image resolution
BATCH_SIZE = 16             # Batch size (reduce to 8 if GPU OOM)
LEARNING_RATE = 0.01        # Initial learning rate
PATIENCE = 10               # Early stopping patience (epochs without improvement)
WORKERS = 1                 # Number of data-loading workers

# Augmentation toggles
USE_MOSAIC = 1.0         # Keep at 100%: Combining 4 images into 1 is excellent for multi-scale object detection.
USE_MIXUP = 0.05         # Reduced from 0.1 -> 0.05: Only 5% blending to avoid destroying potato's rough texture or apple's smoothness.
USE_COPY_PASTE = 0.05    # New (5%): Copy-pasting fruits onto different backgrounds to reduce background dependency.

# 2. HSV Color Tuning (Quantifying shifts instead of just toggling True/False)
# Helps the model recognize fruits effectively in low light or harsh sunlight
HSV_H = 0.015            # Hue: Allows a very slight color shift (1.5%) to avoid turning red apples into green ones.
HSV_S = 0.7              # Saturation: 70% shift, helping the model learn from both vibrant and pale-colored fruits.
HSV_V = 0.4              # Value (Brightness): 40% shift, to cope with images taken in shadows or under lens flare.

# 3. Spatial Deformation (Geometry)
ROTATION_DEGREES = 10.0  # New: Rotate images up to 10 degrees (simulating natural fruit positioning).
FLIP_LR = 0.5            # New: 50% horizontal flip (Highly safe).
FLIP_UD = 0.0

# Training run name (used for output directory)
RUN_NAME = "fruit_disease_v2"

print("=" * 100)
print("TRAINING CONFIGURATION")
print("=" * 100)
print(f"  Model       : {YOLO_MODEL}")
print(f"  Epochs      : {EPOCHS}")
print(f"  Image size  : {IMG_SIZE}")
print(f"  Batch size  : {BATCH_SIZE}")
print(f"  LR          : {LEARNING_RATE}")
print(f"  Patience    : {PATIENCE}")
print(f"  Device      : {DEVICE}")
print(f"  Run name    : {RUN_NAME}")
print("=" * 100)

TRAINING CONFIGURATION
  Model       : D:\in-class\Advance_AI2\advanced_ai_project\models\pretrained\yolov10n.pt
  Epochs      : 100
  Image size  : 800
  Batch size  : 16
  LR          : 0.01
  Patience    : 10
  Device      : cuda
  Run name    : fruit_disease_v2


## 5. Load Pretrained Model & Start Training

In [ ]:
# Load YOLOv10 pretrained model
print(f"⏳ Loading pretrained model: {YOLO_MODEL} ...")
model = YOLO(YOLO_MODEL)
print("✅ Model loaded successfully.\n")

# Start training
print("🚀 Starting training...\n")
results = model.train(
    data=str(DATASET_YAML),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    lr0=LEARNING_RATE,
    patience=PATIENCE,
    workers=WORKERS,
    device=DEVICE,
    project=str(WEIGHTS_DIR),
    name=RUN_NAME,
    exist_ok=True,
    
# ============================================================
# DATA AUGMENTATION CONFIGURATION (Safe Enhancement)
# ============================================================
mosaic=USE_MOSAIC,              # Combines 4 images (default 1.0)
close_mosaic=10,                # Disable mosaic for the last 10 epochs to refine on real images
mixup=USE_MIXUP,                # Pixel blending (0.05)
copy_paste=USE_COPY_PASTE,      # Object copy-pasting (0.05)
hsv_h=HSV_H,                    # Hue shift (0.015)
hsv_s=HSV_S,                    # Saturation shift (0.7)
hsv_v=HSV_V,                    # Brightness shift (0.4)
degrees=ROTATION_DEGREES,       # Random rotation (10.0 degrees)
fliplr=FLIP_LR,                 # Horizontal flip (0.5)
flipud=FLIP_UD,                 # Vertical flip (0.0)

# Logging
verbose=True,
plots=True,                     # REQUIRED: Keep True for YOLO to generate loss/mAP plots for the Technical Report[cite: 1]
)


print("\n" + "=" * 100)
print("🎉 TRAINING COMPLETE")
print("=" * 100)

⏳ Loading pretrained model: D:\in-class\Advance_AI2\advanced_ai_project\models\pretrained\yolov10n.pt ...
✅ Model loaded successfully.

🚀 Starting training...

New https://pypi.org/project/ultralytics/8.4.38 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.0  Python-3.11.9 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 4070, 12282MiB)
engine\trainer: task=detect, mode=train, model=D:\in-class\Advance_AI2\advanced_ai_project\models\pretrained\yolov10n.pt, data=D:\in-class\Advance_AI2\advanced_ai_project\data\processed\dataset.yaml, epochs=100, time=None, patience=10, batch=16, imgsz=800, save=True, save_period=-1, cache=False, device=cuda, workers=1, project=D:\in-class\Advance_AI2\advanced_ai_project\models\weights, name=fruit_disease_v2, exist_ok=True, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False

train: Scanning D:\in-class\Advance_AI2\advanced_ai_project\data\processed\labels\train.cache... 23414 images, 0 backgrounds, 0 corrupt: 100%|██████████| 23414/23414 [00:00<?, ?it/s]

train: WARNING  D:\in-class\Advance_AI2\advanced_ai_project\data\processed\images\train\Carrot__Healthy_freshCarrot (3).jpeg: corrupt JPEG restored and saved
train: WARNING  D:\in-class\Advance_AI2\advanced_ai_project\data\processed\images\train\Carrot__Healthy_freshCarrot (475).jpg: corrupt JPEG restored and saved
train: WARNING  D:\in-class\Advance_AI2\advanced_ai_project\data\processed\images\train\Tomato__Healthy_freshTomato (122).jpg: corrupt JPEG restored and saved



val: Scanning D:\in-class\Advance_AI2\advanced_ai_project\data\processed\labels\val.cache... 4380 images, 0 backgrounds, 0 corrupt: 100%|██████████| 4380/4380 [00:00<?, ?it/s]

val: WARNING  D:\in-class\Advance_AI2\advanced_ai_project\data\processed\images\val\Carrot__Healthy_freshCarrot (491).jpg: corrupt JPEG restored and saved


...

    100/100      4.91G     0.8588      1.104      2.292          9        800: 100%|██████████| 1464/1464 [05:24<00:00,  4.51it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 137/137 [00:23<00:00,  5.89it/s]

                   all       4380       7702       0.77      0.714      0.761      0.694



100 epochs completed in 12.173 hours.
Optimizer stripped from D:\in-class\Advance_AI2\advanced_ai_project\models\weights\fruit_disease_v2\weights\last.pt, 5.8MB
Optimizer stripped from D:\in-class\Advance_AI2\advanced_ai_project\models\weights\fruit_disease_v2\weights\best.pt, 5.8MB

Validating D:\in-class\Advance_AI2\advanced_ai_project\models\weights\fruit_disease_v2\weights\best.pt...
Ultralytics 8.3.0  Python-3.11.9 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 4070, 12282MiB)
YOLOv10n summary (fused): 285 layers, 2,705,336 parameters, 0 gradients, 8.3 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 137/137 [00:21<00:00,  6.24it/s]


                   all       4380       7702      0.771      0.713      0.761      0.694
        Apple__Healthy        365        694      0.877      0.854      0.905      0.852
         Apple__Rotten        438        592      0.951      0.779      0.846       0.77
       Banana__Healthy        299        597      0.787      0.771      0.845      0.676
        Banana__Rotten        419        689      0.754      0.718      0.802      0.643
   Bellpepper__Healthy         91        239      0.618      0.665      0.705      0.641
    Bellpepper__Rotten         88        130      0.612      0.585        0.6      0.522
       Carrot__Healthy         92        523      0.643      0.709       0.71      0.612
        Carrot__Rotten         86        231      0.578      0.463       0.51      0.418
     Cucumber__Healthy         91        127      0.728      0.696      0.788      0.719
      Cucumber__Rotten         88        102      0.755      0.765      0.831       0.72
        Grape__Health

## 6. Results Visualization

Display the training plots that YOLO automatically generates.

In [ ]:
# Path to the training run output directory
RUN_DIR = WEIGHTS_DIR / RUN_NAME

def show_plot(filename: str, title: str):
    """Display a training result image if it exists."""
    path = RUN_DIR / filename
    if path.exists():
        print(f"\n{'─' * 40}")
        print(f"📊 {title}")
        print(f"{'─' * 40}")
        display(Image(filename=str(path), width=800))
    else:
        print(f"⚠️  {title} not found: {path.name}")

# -- Loss & mAP curves --
show_plot("results.png", "Training & Validation Loss / mAP Curves")

# -- Confusion Matrix --
show_plot("confusion_matrix_normalized.png", "Normalized Confusion Matrix")
show_plot("confusion_matrix.png", "Confusion Matrix (raw counts)")

# -- PR Curve --
show_plot("PR_curve.png", "Precision-Recall Curve")

# -- F1 Curve --
show_plot("F1_curve.png", "F1-Confidence Curve")

# -- Sample predictions on validation set --
show_plot("val_batch0_pred.png", "Validation Batch 0 — Predictions")
show_plot("val_batch0_labels.png", "Validation Batch 0 — Ground Truth")

## 7. Evaluate on Test Set

In [ ]:
# Load the best checkpoint from training
best_weights = RUN_DIR / "weights" / "best.pt"
assert best_weights.exists(), f"Best weights not found at {best_weights}"

print(f"⏳ Loading best weights: {best_weights}")
best_model = YOLO(str(best_weights))

# Run evaluation on the test split
print("\n🔍 Evaluating on the TEST set...\n")
test_metrics = best_model.val(
    data=str(DATASET_YAML),
    split="test",
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    verbose=True,
)

print("\n" + "=" * 50)
print("TEST SET RESULTS")
print("=" * 50)
print(f"  mAP@50      : {test_metrics.box.map50:.4f}")
print(f"  mAP@50-95   : {test_metrics.box.map:.4f}")
print(f"  Precision   : {test_metrics.box.mp:.4f}")
print(f"  Recall      : {test_metrics.box.mr:.4f}")
print("=" * 50)

## 8. Export Best Weights

Copy the best checkpoint to the project's standard weights directory for easy access by other modules (`src/detection/detector.py`).

In [ ]:
# Copy best.pt to the top-level weights directory
export_path = WEIGHTS_DIR / "best.pt"
shutil.copy(best_weights, export_path)
print(f"✅ Best weights exported to: {export_path}")

# Also copy last.pt as a backup
last_weights = RUN_DIR / "weights" / "last.pt"
if last_weights.exists():
    shutil.copy(last_weights, WEIGHTS_DIR / "last.pt")
    print(f"✅ Last weights exported to: {WEIGHTS_DIR / 'last.pt'}")

print("\n" + "=" * 50)
print("🎉 ALL DONE — Model is ready for inference!")
print(f"   Use weights at: {export_path}")
print("=" * 50)